# Bằng chứng: V-JEPA 2 cắt clip EK100 ở đâu so với hành động (chỉ cần CPU)

Notebook này cho hai loại bằng chứng:

1. **Chạy chính code của Meta** (`vjepa2/evals/action_anticipation_frozen/epickitchens.py`, hàm `decode_videos_to_clips.run`, commit
   `204698b`) với một `VideoReader` giả chỉ ghi lại các frame mà code yêu cầu. In ra: mỗi hành động có nhãn cho đúng một clip, và clip
   nằm ở đâu so với hành động.
2. **Hình thật** của 32 frame (hiện 8) mà mô hình nhìn thấy, ở hai thiết lập của cùng công thức trong code Meta:
   `af = int(sf·ap + (1 − ap)·ef − 1 s·fps)`, clip = 32 frame, bước `int(fps / 8)`, kết thúc ngay trước `af`.
   - `ap = 0` (config Meta phát hành): clip kết thúc **1 s trước khi hành động kết thúc**.
   - `ap = 1` (định nghĩa EK100): clip kết thúc **1 s trước khi hành động bắt đầu**.
   Frame có viền đỏ = nằm trong khoảng thời gian của chính hành động cần đoán.

**Cài đặt:** Accelerator None, Internet On. *Add Input*: bản 292p của val (`reencode-0`, `reencode-1`): mọi frame được giữ nguyên nên
chỉ số frame giống video gốc.

In [ ]:
VIDEO = "P01_11"
NARRATION_IDS = ["P01_11_2", "P01_11_6", "P01_11_9"]   # take pizza / take pizza slice / fold paper
BRANCH = "direction2-balanced-head"
!git clone -q https://github.com/1Tyson/Testing-And-Research-VJEPA.git -b $BRANCH /tmp/repo
!git clone -q https://github.com/facebookresearch/vjepa2.git /tmp/vjepa2 && git -C /tmp/vjepa2 checkout -q 204698b45b3712590f06245fbfba32d3be539812
!git clone -q --depth 1 https://github.com/epic-kitchens/epic-kitchens-100-annotations.git /tmp/ek100-ann
!pip install -q decord webdataset

In [ ]:
# 1. Meta's own sampling code, run on the real annotations (frame indices recorded, no video decoded)
!python /tmp/repo/scripts/show_official_clip_window.py --vjepa2 /tmp/vjepa2 --annotations /tmp/ek100-ann --video $VIDEO --n 12
# the lines of Meta's code that place the clip:
!sed -n 114,137p /tmp/vjepa2/evals/action_anticipation_frozen/epickitchens.py
!grep -n "val_anticipation_point\|anticipation_time_sec" /tmp/vjepa2/evals/action_anticipation_frozen/eval.py | head -4
!grep -n -A2 "anticipation_time_sec:\|frames_per_clip\|frames_per_second\|anticipation_point" /tmp/vjepa2/configs/eval/vitl/ek100.yaml

In [ ]:
# 2. The frames the model actually sees (292p copy: same frame indices as the original video)
import glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from decord import VideoReader, cpu
path = [p for p in glob.glob(f"/kaggle/input/**/{VIDEO}.MP4", recursive=True)][0]
vr = VideoReader(path, ctx=cpu(0)); fps = vr.get_avg_fps(); step = int(fps / 8)
ann = pd.read_csv("/tmp/ek100-ann/EPIC_100_validation.csv").set_index("narration_id")

def clip(sf, ef, ap):           # same two lines as Meta's code (af, indices), at = 1 s
    af = int(sf * ap + (1 - ap) * ef - int(1.0 * fps))
    idx = np.arange(af - 32 * step, af, step); idx[idx < 0] = 0
    return idx

for nid in NARRATION_IDS:
    r = ann.loc[nid]; sf, ef = int(r.start_frame), int(r.stop_frame)
    rows = [("Meta (ap = 0): kết thúc 1 s trước KẾT THÚC hành động", clip(sf, ef, 0.0)),
            ("EK100 (ap = 1): kết thúc 1 s trước KHI BẮT ĐẦU hành động", clip(sf, ef, 1.0)),
            ("Tham khảo: chính hành động (start → stop)", np.linspace(sf, ef, 8).astype(int))]
    fig, axs = plt.subplots(3, 8, figsize=(16, 6.2))
    fig.suptitle(f'{nid}: "{r.narration}"  —  hành động {sf / fps:.2f}–{ef / fps:.2f} s', fontsize=13)
    for row, (title, idx) in enumerate(rows):
        show = idx if len(idx) == 8 else idx[::4]          # 8 of the 32 frames
        frames = vr.get_batch([int(i) for i in show]).asnumpy()
        for k, (f, im) in enumerate(zip(show, frames)):
            ax = axs[row, k]; ax.imshow(im); ax.set_xticks([]); ax.set_yticks([])
            inside = sf <= f <= ef
            for s in ax.spines.values():
                s.set_edgecolor("red" if inside else "#bbbbbb"); s.set_linewidth(4 if inside else 1)
            ax.set_xlabel(f"{f / fps:.2f} s", fontsize=9, color="red" if inside else "black")
        axs[row, 0].set_ylabel(title.split(":")[0], fontsize=10)
        axs[row, 0].set_title(title, loc="left", fontsize=10)
    plt.tight_layout(); plt.savefig(f"/kaggle/working/clip_{nid}.png", dpi=110); plt.show()

**Đọc hình:** hàng 1 là thứ mô hình thấy theo code Meta, hàng 2 theo định nghĩa EK100, hàng 3 là chính hành động. Viền đỏ = frame
thuộc hành động cần đoán. Ở code Meta, các hành động dài hơn 1 s (84% val) luôn có frame viền đỏ; ở EK100 thì không bao giờ.